# Pitfalls in fitting 2D histograms

A colleague asked me about a seeming simple problem of fitting a line model to a 2D data distribution and why their solution was biased. The reasons why the original approach failed are surprisingly subtle. In fact, when I tackled it, I first made a mistake as well! So I think it is worthwhile to present this case to learn from our mistakes.

Some years back, I wrote a paper about this very problem 
[H. Dembinski et al., Astropart.Phys. 73 (2016) 44-51](https://doi.org/10.1016/j.astropartphys.2015.08.001), which contains the exact solution for the slightly more complex problem of fitting calibration data from two experiments, and a clever approximation that can be computed much faster than the exact solution.

**Bottom line**: When in doubt, really write down the exact likelihood for the data that you have. The likelihood, when derived strictly from first principles, is always correct (= asymptotically unbiased). Any kind of ad hoc approximations will fail you unless they can be derived from the exact likelihood.

## Setup

The data that we want to fit is two-dimensional. We have a variable $y$ which is an affine function of $x$ (can be modelled by a line). The observed value of $y$ is affected by random scatter that follows a normal distribution. We visualize this with a histogram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import boost_histogram as bh

rng = np.random.default_rng(1)
n_samples = 1_000_000
data_x = rng.uniform(-10, 10, n_samples)

true_slope = 2.0
true_intercept = 1.0
true_sigma = 5.0


def line(x, intercept, slope):
    return intercept + slope * x


true_y = line(data_x, true_intercept, true_slope)
data_y = rng.normal(true_y, true_sigma)

hist = bh.Histogram(bh.axis.Regular(50, -10, 10), bh.axis.Regular(50, -10, 10))
hist.fill(data_x, data_y)

x_edges = hist.axes[0].edges
y_edges = hist.axes[1].edges
counts = hist.values()

plt.pcolormesh(x_edges, y_edges, counts.T)

plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(label="counts");

We make a custom plotter to visualize the Minuit fits below.

In [ ]:
def plotter(args):
    plt.pcolormesh(x_edges, y_edges, counts.T)
    x_fit = np.linspace(x_edges[0], x_edges[-1])
    # ignore sigma parameter
    y_fit = line(x_fit, *args[:2])
    plt.plot(x_fit, y_fit, color="red", label="Fit")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(x_edges[0], x_edges[-1])
    plt.ylim(y_edges[0], y_edges[-1])
    plt.colorbar(label="counts")

## Case 1: Fit complete sample

If you have the complete sample of $(x, y)$ pairs at hand which is not affected by any cuts, you can do an unbinned maximum-likelihood fit. When writing down the PDF model, you need to declare that the PDF of the $y$ values is conditional on the $x$ values. This case is covered in more detail in the tutorial *Fit PDF with conditional variable*.

In [ ]:
from scipy.stats import norm
from iminuit import Minuit
from iminuit import cost


# this is a nice opportunity where can directly model the logpdf
def model(xy, intercept, slope, sigma):
    x, y = xy
    mu = line(x, intercept, slope)
    # cannot use norm.pdf from numba_stats here, because it is not vectorized in mu
    return norm.logpdf(y, mu, sigma)


nll = cost.UnbinnedNLL((data_x, data_y), model, log=True)
m = Minuit(nll, true_intercept, true_slope, true_sigma)
m.plotter = plotter
m.limits["sigma"] = (0, None)
m.migrad()

We nicely recover the true values.

Alternatively, one can also fit these data with the LeastSquares class. This works because the variance of the $y$ values around the model line is constant. The LeastSquares class expects that we pass an estimate of the uncertainty for each data point. We use the true uncertainty here.

In [ ]:
lsq = cost.LeastSquares(data_x, data_y, true_sigma, line)
m = Minuit(lsq, true_intercept, true_slope)
m.plotter = plotter
m.migrad()

A caveat of these two approaches is that we need to model the variance of the $y$ values. If that is difficult, it is better to empirically estimate the variance. The `boost-histogram` package offers the `Mean` storage for that purpose. It generates a "profile" in ROOT language: the mean and variance of the $y$ values are computed in bins of $x$. We can then pass these estimates to the LeastSquares class. This will also speed up the fit considerably.

In [ ]:
profile = bh.Histogram(bh.axis.Regular(20, -10, 10), storage=bh.storage.Mean())
profile.fill(data_x, sample=data_y)

lsq = cost.LeastSquares(
    profile.axes[0].centers, profile.values(), profile.variances() ** 0.5, line
)
m = Minuit(lsq, true_intercept, true_slope)
m.migrad()

## Case 2: Fit incomplete histogram

We often have the situation that the full sample is not available, and we have a histogram instead. Maybe it is more convenient to use a histogram, because the sample is very large. Maybe there are cuts in place, explicit or implicit, so that the sample is not complete anyway. Then we should fit the histogram.

If we try to treat this like a weighted least-squares problem, we find that the fit result is strongly biased.

In [ ]:
x_centers = hist.axes[0].centers
y_centers = hist.axes[1].centers
x, y = np.meshgrid(x_centers, y_centers, indexing="ij")
counts = hist.values()


def cost_function(intercept, slope):
    ym = line(x, intercept, slope)
    residuals = (y - ym) ** 2 / true_sigma**2
    weights = counts
    return np.sum(residuals * weights)


m = Minuit(cost_function, true_intercept, true_slope)
m.plotter = plotter
m.migrad()

While this approach could work in theory if the histogram would fully cover the whole sample, we get a biased result because it does not. The subsample contained in the histogram is cut off at $y < -10$ and $y > 10$. This information is not present in our cost function.

If we consider this problem from first principles, we need to find the likelihood function for this problem. We need a model that predicts the density in each cell of the histogram. The actual contents are Poisson-distributed around this density.

The easiest way to fit such a histogram is to use the `ExtendedBinnedNLL` cost function from iminuit.

`ExtendedBinnedNLL` wants a model that predicts the scaled CDF, but it also accepts a density model (scaled PDF) as an approximation. We have narrow bins here so a density approximation works well. The density is easier to write down. We have a uniform distribution along $x$, and a normal distribution along $y$ that is conditional on $x$. We can multiply the two densities, while the analytical expression for the CDF would be more complicated since the integrals are not independent.

In [ ]:
from scipy.stats import uniform


def model(xy, intercept, slope, sigma, ntot):
    x, y = xy
    ym = line(x, intercept, slope)
    return ntot * uniform.pdf(x, -10, 20) * norm.pdf(y, ym, sigma)


# beware: it does not work with BinnedNLL, because that assumes all events are observed
cost_fn = cost.ExtendedBinnedNLL(
    counts, (x_edges, y_edges), model, use_pdf="approximate"
)

m = Minuit(
    cost_fn,
    intercept=true_intercept,
    slope=true_slope,
    sigma=true_sigma,
    ntot=np.sum(counts),
)
m.plotter = plotter
m.migrad()

The fit is unbiased again. We lost a bit of accuracy compared to case 1, that is expected because the fitted sample here is smaller.

Note that using `BinnedNLL` instead of `ExtendedBinnedNLL` leads to the same biased result as the initial attempt with the least-squares method.

In [ ]:
# beware: it does not work with BinnedNLL, because that assumes all events are observed
cost_fn = cost.BinnedNLL(counts, (x_edges, y_edges), model, use_pdf="approximate")

m = Minuit(
    cost_fn,
    intercept=true_intercept,
    slope=true_slope,
    sigma=true_sigma,
    ntot=np.sum(counts),
)
m.fixed["ntot"] = True
m.plotter = plotter
m.migrad()

This is biased, because we did not normalize our model over this restricted parameter space.

If we properly normalize our model, however, we also get an unbiased result.

In [ ]:
def model(xy, intercept, slope, sigma):
    x, y = xy
    ym = line(x, intercept, slope)
    # we properly normalize the normal distribution over the histogram space
    p = norm.pdf(y, ym, sigma) / (norm.cdf(10, ym, sigma) - norm.cdf(-10, ym, sigma))
    return uniform.pdf(x, -10, 20) * p


# beware: it does not work with BinnedNLL, because that assumes all events are observed
cost_fn = cost.BinnedNLL(counts, (x_edges, y_edges), model, use_pdf="approximate")

m = Minuit(cost_fn, intercept=true_intercept, slope=true_slope, sigma=true_sigma)
m.plotter = plotter
m.migrad()